# Pipeline de Inferencia Batch

El propósito de este notebook es cargar un modelo de deep learning previamente entrenado y utilizarlo para generar predicciones sobre un nuevo conjunto de datos (inferencia batch). Este proceso es típico en escenarios donde las predicciones no se necesitan en tiempo real, sino que se procesan en lotes periódicamente (e.g., diariamente, semanalmente).

## Cargar Bibliotecas y Modelo Entrenado

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf # Usado para tf.keras.models
from tensorflow.keras.models import load_model

In [ ]:
# Ruta al modelo entrenado
model_path = '../models/bank_marketing_model.keras'
model_loaded_successfully = False
model = None

# Cargar el modelo Keras
try:
    model = load_model(model_path)
    model_loaded_successfully = True
    print(f"Modelo cargado exitosamente desde: {model_path}")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    print("Asegúrese de que el notebook '03_training_pipeline.ipynb' se haya ejecutado y el modelo se haya guardado correctamente.")

In [ ]:
if model_loaded_successfully and model is not None:
    print("\nResumen del modelo cargado:")
    model.summary()
else:
    print("\nEl modelo no está cargado, no se puede mostrar el resumen.")

## Cargar Datos para Inferencia

Para la inferencia, necesitamos cargar datos que tengan la misma estructura y preprocesamiento que los datos utilizados para entrenar el modelo. En este ejemplo, utilizaremos el conjunto de prueba (`X_test`) que fue procesado y guardado por el notebook `02_feature_pipeline.ipynb`.

En un escenario de producción, estos podrían ser nuevos datos de clientes que han pasado por el mismo pipeline de ingeniería de características (codificación, escalado, etc.).

In [ ]:
# Ruta a los datos de prueba (características)
test_features_file = '../processed_data/test_features.csv'
data_for_inference_loaded = False
X_inference = pd.DataFrame()

# Cargar los datos
try:
    X_inference = pd.read_csv(test_features_file)
    data_for_inference_loaded = True
    print(f"Datos para inferencia cargados exitosamente desde: {test_features_file}")
    print(f"Forma de los datos cargados: {X_inference.shape}")
except FileNotFoundError:
    print(f"Error: El archivo '{test_features_file}' no se encontró.")
    print("Asegúrese de que el notebook '02_feature_pipeline.ipynb' se haya ejecutado y los datos procesados se hayan guardado.")
except Exception as e:
    print(f"Ocurrió un error al cargar los datos para inferencia: {e}")

In [ ]:
if data_for_inference_loaded and not X_inference.empty:
    print("\nPrimeras 5 filas de los datos cargados para inferencia (X_inference):")
    display(X_inference.head())
else:
    print("\nNo se cargaron datos para inferencia o el DataFrame está vacío.")

## Generar Predicciones

Con el modelo cargado y los datos listos, podemos generar las predicciones. El método `model.predict()` devuelve las probabilidades de pertenencia a la clase positiva (en este caso, la probabilidad de que un cliente suscriba el depósito).

Luego, convertiremos estas probabilidades en clases binarias (0 o 1) aplicando un umbral, comúnmente 0.5. Si la probabilidad es > 0.5, se clasifica como 1 ('yes'); de lo contrario, como 0 ('no').

In [ ]:
predictions_proba = None
if model_loaded_successfully and model is not None and data_for_inference_loaded and not X_inference.empty:
    print("Generando predicciones (probabilidades)...")
    predictions_proba = model.predict(X_inference)
    print(f"Forma de las predicciones de probabilidad: {predictions_proba.shape}")
    print("\nEjemplo de primeras 5 predicciones de probabilidad:")
    print(predictions_proba[:5])
else:
    print("No se pueden generar predicciones. Modelo o datos no cargados correctamente.")

In [ ]:
predictions_binary = None
if predictions_proba is not None:
    # Aplicar umbral para convertir probabilidades a clases binarias
    threshold = 0.5
    predictions_binary = (predictions_proba > threshold).astype(int)
    print(f"\nPredicciones convertidas a clases binarias (0 o 1) usando un umbral de {threshold}.")
    print("\nEjemplo de primeras 5 predicciones binarias:")
    print(predictions_binary[:5])
else:
    print("\nNo hay predicciones de probabilidad para convertir a binarias.")

## Guardar Predicciones (Conceptual)

En un entorno de producción, las predicciones generadas se guardarían en un sistema persistente para su uso posterior. Esto podría ser:
*   Una tabla en una **base de datos** relacional o NoSQL.
*   Un archivo (e.g., CSV, Parquet) en un sistema de almacenamiento como **Amazon S3, Google Cloud Storage, o Azure Blob Storage**.
*   Enviadas a un **sistema de mensajería** (e.g., Kafka) para ser consumidas por otros servicios.

Para este ejemplo, guardaremos las predicciones en un archivo CSV localmente.

In [ ]:
predictions_dir = "../predictions/"
predictions_saved = False

if predictions_binary is not None:
    try:
        os.makedirs(predictions_dir, exist_ok=True)
        print(f"Directorio '{predictions_dir}' listo o ya existente.")
    except OSError as e:
        print(f"Error al crear el directorio {predictions_dir}: {e}")
    
    # Crear un DataFrame para las predicciones
    # Si tuviéramos IDs para cada instancia en X_inference, los incluiríamos aquí.
    # Por ejemplo, si X_inference tuviera un índice reseteado que sirva como ID temporal:
    # df_predictions = pd.DataFrame({'instance_id': X_inference.index, 'prediction_proba': predictions_proba.flatten(), 'prediction_class': predictions_binary.flatten()})
    # Para este caso, solo guardaremos las predicciones binarias y de probabilidad.
    df_predictions = pd.DataFrame({
        'prediction_probability_yes': predictions_proba.flatten(), 
        'predicted_class': predictions_binary.flatten()
    })
    
    # Mapear 0/1 a 'no'/'yes' para mayor claridad si se desea
    df_predictions['predicted_label'] = df_predictions['predicted_class'].map({0: 'no', 1: 'yes'})

    # Guardar las predicciones en un archivo CSV
    predictions_file_path = os.path.join(predictions_dir, 'batch_predictions.csv')
    try:
        df_predictions.to_csv(predictions_file_path, index=False)
        print(f"Predicciones guardadas exitosamente en: {predictions_file_path}")
        print("\nPrimeras filas de las predicciones guardadas:")
        display(df_predictions.head())
        predictions_saved = True
    except Exception as e:
        print(f"Error al guardar las predicciones en CSV: {e}")
else:
    print("No hay predicciones binarias para guardar.")

Si `predictions_saved` es `True`, las predicciones están ahora listas para ser consumidas por otros procesos o para análisis posteriores. Por ejemplo, podrían ser cargadas en un dashboard, utilizadas para segmentar clientes, o para iniciar acciones de marketing específicas.

## Resumen

En este notebook, hemos demostrado el proceso de inferencia batch:
1.  **Carga del Modelo:** Se cargó un modelo Keras previamente entrenado desde el disco.
2.  **Carga de Datos:** Se cargaron datos preprocesados listos para la inferencia (simulando un nuevo lote de datos).
3.  **Generación de Predicciones:** El modelo generó predicciones de probabilidad, que luego se convirtieron a clases binarias.
4.  **Guardado de Predicciones:** Las predicciones resultantes (probabilidades y clases) se guardaron en un archivo CSV, simulando cómo se almacenarían en un sistema de producción.

Este pipeline es fundamental para aplicar modelos de machine learning a escala y obtener valor de ellos de forma continua.